# Final Confidence Score — Trip-Level Analysis

**Objective:** Extract key scoring columns from the trip-level dataset, enrich each record with the vendor label, and compute a composite **Final Confidence Score** that blends data-quality and execution-reliability signals.

$$
\text{Final Confidence Score} = 0.4 \times \text{Trip Data Quality Score} + 0.6 \times \text{Trip Execution Reliability Score}
$$

---

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)
print("Libraries loaded successfully.")

## 1 — Load the Raw Data

In [ ]:
# Load both datasets
PATH = "<INPUT_DATA_DIR>/"
trips_raw = pd.read_csv(PATH + "final_trip_score_v2.csv")
vendors    = pd.read_csv(PATH + "vendors.csv")

print(f"Trips  : {trips_raw.shape[0]:,} rows  ×  {trips_raw.shape[1]} columns")
print(f"Vendors: {vendors.shape[0]:,} rows  ×  {vendors.shape[1]} columns")

In [ ]:
# Quick peek at the relevant columns
print("── Trips (first 5 rows, selected columns) ──")
display(trips_raw[["trip_id", "vendor_id",
                    "Trip Execution Reliability Score",
                    "Trip Data Quality Score"]].head())

print("\n── Vendors ──")
display(vendors.head())

## 2 — Extract Required Columns

In [ ]:
# Keep only the four columns specified in the brief
keep_cols = [
    "trip_id",
    "vendor_id",
    "Trip Execution Reliability Score",
    "Trip Data Quality Score"
]

trips = trips_raw[keep_cols].copy()
print(f"Extracted dataframe: {trips.shape[0]:,} rows × {trips.shape[1]} columns")
trips.head()

## 3 — Join Vendor Labels

In [ ]:
# Left-join to bring in the vendor_label column
trips = trips.merge(
    vendors[["vendor_id", "vendor_label"]],
    on="vendor_id",
    how="left"
)

# Check for any unmatched vendors
unmatched = trips["vendor_label"].isna().sum()
print(f"Unmatched vendor_ids: {unmatched}")

trips.head()

## 4 — Compute the Final Confidence Score

$$
\text{Final Confidence Score} = 0.4 \times \text{Trip Data Quality Score} + 0.6 \times \text{Trip Execution Reliability Score}
$$

In [ ]:
# Weighted composite score
trips["Final Confidence Score"] = (
    0.4 * trips["Trip Data Quality Score"]
  + 0.6 * trips["Trip Execution Reliability Score"]
)

# Round for readability
trips["Final Confidence Score"] = trips["Final Confidence Score"].round(2)

# Reorder columns for a clean final table
trips = trips[[
    "trip_id",
    "vendor_id",
    "vendor_label",
    "Trip Execution Reliability Score",
    "Trip Data Quality Score",
    "Final Confidence Score"
]]

print(f"Final dataframe: {trips.shape[0]:,} rows × {trips.shape[1]} columns")
trips.head(10)

## 5 — Summary Statistics

In [ ]:
# Descriptive statistics for the three score columns
score_cols = [
    "Trip Execution Reliability Score",
    "Trip Data Quality Score",
    "Final Confidence Score"
]

trips[score_cols].describe()

In [ ]:
# Per-vendor average scores
vendor_summary = (
    trips
    .groupby("vendor_label")[score_cols]
    .agg(["mean", "median", "count"])
    .round(2)
)

vendor_summary

## 6 — Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, col in zip(axes, score_cols):
    ax.hist(trips[col].dropna(), bins=30, edgecolor="white", alpha=0.85)
    ax.set_title(col, fontsize=10, fontweight="bold")
    ax.set_xlabel("Score")
    ax.axvline(trips[col].mean(), color="red", linestyle="--", label=f"Mean {trips[col].mean():.1f}")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Trip Count")
fig.suptitle("Distribution of Trip-Level Scores", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Mean Final Confidence Score by vendor
vendor_mean = (
    trips.groupby("vendor_label")["Final Confidence Score"]
    .mean()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
vendor_mean.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Mean Final Confidence Score")
ax.set_title("Average Final Confidence Score by Vendor", fontweight="bold")
ax.axvline(trips["Final Confidence Score"].mean(), color="red",
           linestyle="--", label=f'Overall Mean ({trips["Final Confidence Score"].mean():.1f})')
ax.legend()
plt.tight_layout()
plt.show()

## 7 — Export Results

In [ ]:
output_path = PATH + "trips_final_confidence_scores.csv"
trips.to_csv(output_path, index=False)
print(f"Saved {trips.shape[0]:,} rows → {output_path}")